<h1 dir=rtl align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
پروژه کرونو-ویژن
</font>
</h1>

<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
دانلود داده ها
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
فایل ویدئو ها را میتوانید از <a href="https://drive.google.com/file/d/17XLIUXqzWgCUVFF5aWY_UEcFftKFDwcP/view?usp=drive_link">لینک</a> دانلود کنید. شما باید طبق چیزی که در صورت سوال گفته شده، اشیا متحرک را سگمنت کنید.
</font>
</p>


<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
نوشتن کدها
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
شما در نهایت باید همراه با پوشه و فولدر ویدئوهای سگمنت شده فایل کد خود را نیز در کوئرا بارگذاری کنید. پس توصیه میشود از همین فایل نوت بوک در جهت نوشتن کدها استفاده کنید یا اینکه، در نهایت فایل های خود را در این نوت بوک قرار داده و سابمیت کنید. یا در نهایت خودتان دستی فایلی به نام <code>notebook.ipynb</code> بسازید و سابمیت کنید.
</font>
</p>


<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
<b>سلول جواب‌ساز</b>
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
    برای ساخته‌شدن فایل <code>result.zip</code> سلول زیر را اجرا کنید. توجه داشته باشید که پیش از اجرای سلول زیر تغییرات اعمال شده در نت‌بوک را ذخیره کرده باشید (<code>ctrl+s</code>) در غیر این صورت، در پایان مسابقه نمره شما به صفر تغییر خواهد کرد.
    <br>
    همچنین اگر از کولب برای اجرای این فایل نوت‌بوک استفاده می‌کنید، قبل از ارسال فایل <code>result.zip</code>، آخرین نسخه‌ی نوت‌بوک خود را دانلود کرده و داخل فایل ارسالی قرار دهید.
</font>

In [ ]:
# !pip install ultralytics opencv-python numpy pandas
from tqdm.auto import tqdm
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
import os
from pathlib import Path

VIDEOS_DIR = Path('videos')           
SUBMISSION_DIR = Path('submission')     

MODEL_NAME = 'yolov8l-seg.pt' 
PERSON_CLASS_ID = 0            

STILL_WINDOW_N = 15           
MOVEMENT_THRESHOLD = 5      
STILL_COLOR = (255, 255, 255) 

TRACK_COLORS_MAP = {
    1: (0, 0, 255),   
    2: (0, 255, 0),   
    3: (0, 255, 255),
}


In [ ]:
def get_centroid(bbox):

    x1, y1, x2, y2 = bbox
    return int((x1 + x2) // 2), int((y1 + y2) // 2)

def calculate_displacement(group, window_n, threshold):
  
    if len(group) < window_n:
        group['is_still'] = False
        return group

    x_t, y_t = group['centroid_x'].values, group['centroid_y'].values

    x_t_minus_n = np.roll(x_t, window_n)
    y_t_minus_n = np.roll(y_t, window_n)

    # فاصله اقلیدسی
    distances = np.sqrt((x_t - x_t_minus_n)**2 + (y_t - y_t_minus_n)**2)
    distances[:window_n] = threshold + 1

    group['is_still'] = distances < threshold
    return group


In [ ]:
def process_video_to_frames(video_path, output_dir, stillness_lookup):
    model = YOLO(MODEL_NAME)
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        print(f"❌ خطا: باز کردن ویدیوی {video_path.name} ممکن نیست.")
        return

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    output_dir.mkdir(parents=True, exist_ok=True)

    id_color_map = {}
    color_counter = 1
    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        frame_count += 1

        results = model.track(frame, persist=True, tracker='botsort.yaml',
                              classes=PERSON_CLASS_ID, conf=0.5, verbose=False)

        output_frame = np.zeros((frame_height, frame_width, 3), dtype=np.uint8)

        if results[0].boxes.id is not None and results[0].masks is not None:
            track_ids = results[0].boxes.id.cpu().numpy().astype(int)
            masks = results[0].masks.data.cpu().numpy()

            for mask, track_id in zip(masks, track_ids):

                if track_id not in id_color_map:
                    id_color_map[track_id] = TRACK_COLORS_MAP.get(color_counter, STILL_COLOR)
                    if color_counter <= 3: color_counter += 1

                is_still = stillness_lookup.get((frame_count, track_id), False)
                segment_color = STILL_COLOR if is_still else id_color_map[track_id]

                mask_resized = cv2.resize(mask.astype(np.uint8), (frame_width, frame_height))
                color_mask = np.zeros((frame_height, frame_width, 3), dtype=np.uint8)
                color_mask[mask_resized > 0] = segment_color
                output_frame = np.where(color_mask > 0, color_mask, output_frame)

        cv2.imwrite(str(output_dir / f"{video_path.stem}_{frame_count}.png"), output_frame)

    cap.release()
    print(f"✅ {frame_count} فریم در {output_dir.name} ذخیره شد.")


In [ ]:
from tqdm.auto import tqdm

def main_pipeline():
    SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
    video_files = list(VIDEOS_DIR.glob('*.mp4'))

    if not video_files:
        print("❌ پوشه ویدیوها خالی است!")
        return

    print(f"🔥 شروع پردازش {len(video_files)} ویدیو...")
    model = YOLO(MODEL_NAME)
    temp_tracking_data = []

    for video_path in video_files:
        print(f"\n⏳ استخراج داده مسیر: {video_path.name}")
        cap = cv2.VideoCapture(str(video_path))

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_count = 0

        with tqdm(total=total_frames, desc="Processing Frames", unit="frame") as pbar:
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret: break
                frame_count += 1

                results = model.track(frame, persist=True, tracker='botsort.yaml',
                                      classes=PERSON_CLASS_ID, conf=0.5, verbose=False,
                                      half=True, device=0)

                if results[0].boxes.id is not None:
                    track_ids = results[0].boxes.id.cpu().numpy().astype(int)
                    bboxes = results[0].boxes.xyxy.cpu().numpy().astype(int)

                    for track_id, bbox in zip(track_ids, bboxes):
                        cx, cy = get_centroid(bbox)
                        temp_tracking_data.append({
                            'video_name': video_path.stem,
                            'frame_id': frame_count, 'track_id': track_id,
                            'centroid_x': cx, 'centroid_y': cy
                        })

                pbar.update(1)

        cap.release()

    if not temp_tracking_data: return
    print("\n🧠 در حال تحلیل وضعیت سکون...")
    df = pd.DataFrame(temp_tracking_data)
    df_analyzed = df.groupby(['video_name', 'track_id'], group_keys=False).apply(
        lambda g: calculate_displacement(g, STILL_WINDOW_N, MOVEMENT_THRESHOLD)
    )
    stillness_lookup = df_analyzed.set_index(['video_name', 'frame_id', 'track_id'])['is_still'].to_dict()

    print("\n🎨 شروع نقاشی و ذخیره فریم‌های نهایی...")

    for video_path in tqdm(video_files, desc="Saving Final Videos", unit="video"):
        v_name = video_path.stem

        v_lookup = {(fid, tid): st for (vn, fid, tid), st in stillness_lookup.items() if vn == v_name}
        process_video_to_frames(video_path, SUBMISSION_DIR / v_name, v_lookup)

    print("\n🎉 فرآیند با موفقیت تکمیل شد.")
    
main_pipeline()


🔥 شروع پردازش 29 ویدیو...

⏳ استخراج داده مسیر: O_SM_04.mp4


Processing Frames:   0%|          | 0/425 [00:00<?, ?frame/s]

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 185ms
Prepared 1 package in 31ms
Installed 1 package in 2ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


⏳ استخراج داده مسیر: O_CL_02.mp4


Processing Frames:   0%|          | 0/425 [00:00<?, ?frame/s]


⏳ استخراج داده مسیر: O_SU_02.mp4


Processing Frames:   0%|          | 0/400 [00:00<?, ?frame/s]

... (52 outputs left out)


🧠 در حال تحلیل وضعیت سکون...

🎨 شروع نقاشی و ذخیره فریم‌های نهایی...


/tmp/ipykernel_6567/2721984880.py:56: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_analyzed = df.groupby(['video_name', 'track_id'], group_keys=False).apply(


Saving Final Videos:   0%|          | 0/29 [00:00<?, ?video/s]

✅ 425 فریم در O_SM_04 ذخیره شد.
✅ 425 فریم در O_CL_02 ذخیره شد.
✅ 400 فریم در O_SU_02 ذخیره شد.
✅ 300 فریم در I_SM_01 ذخیره شد.
✅ 450 فریم در I_MB_01 ذخیره شد.
✅ 275 فریم در I_BS_02 ذخیره شد.
✅ 300 فریم در I_SM_05 ذخیره شد.
✅ 300 فریم در I_SM_06 ذخیره شد.
✅ 850 فریم در O_SN_02 ذخیره شد.
✅ 300 فریم در I_MC_01 ذخیره شد.
✅ 250 فریم در I_OC_01 ذخیره شد.
✅ 300 فریم در I_SI_01 ذخیره شد.
✅ 425 فریم در O_SM_12 ذخیره شد.
✅ 300 فریم در I_SM_12 ذخیره شد.
✅ 275 فریم در I_BS_01 ذخیره شد.
✅ 300 فریم در I_SI_02 ذخیره شد.
✅ 225 فریم در O_CL_01 ذخیره شد.
✅ 500 فریم در O_SN_01 ذخیره شد.
✅ 375 فریم در O_RA_02 ذخیره شد.
✅ 250 فریم در O_SU_01 ذخیره شد.
✅ 525 فریم در I_CA_02 ذخیره شد.
✅ 250 فریم در I_OC_02 ذخیره شد.
✅ 525 فریم در I_IL_02 ذخیره شد.
✅ 425 فریم در O_MC_01 ذخیره شد.
✅ 300 فریم در I_SM_09 ذخیره شد.
✅ 300 فریم در I_SM_03 ذخیره شد.
✅ 425 فریم در O_SM_02 ذخیره شد.
✅ 350 فریم در I_CA_01 ذخیره شد.
✅ 250 فریم در I_MC_02 ذخیره شد.

🎉 فرآیند با موفقیت تکمیل شد.


In [9]:
import os
import zipfile

if not os.path.exists(os.path.join(os.getcwd(), 'notebook.ipynb')):
    %notebook -e notebook.ipynb

SUBMISSION_DIR = 'submission'
NOTEBOOK_FILE = 'notebook.ipynb'
OUTPUT_ZIP_FILE = 'result.zip'

%notebook -e {NOTEBOOK_FILE}
if not os.path.isdir(SUBMISSION_DIR):
    print(f"خطا: پوشه '{SUBMISSION_DIR}' یافت نشد.")
    print("لطفاً ابتدا نتایج خود را در این پوشه تولید کرده و سپس این سلول را اجرا کنید.")
else:
    print(f"پوشه '{SUBMISSION_DIR}' با موفقیت یافت شد.")
    print("شروع فرآیند فشرده‌سازی...")

    try:
        with zipfile.ZipFile(OUTPUT_ZIP_FILE, 'w', zipfile.ZIP_DEFLATED) as zipf:
            # الف) اضافه کردن پوشه submission به همراه تمام محتویات آن
            print(f"در حال اضافه کردن پوشه '{SUBMISSION_DIR}'...")
            for root, dirs, files in os.walk(SUBMISSION_DIR):
                for file in files:
                    file_path = os.path.join(root, file)
                    # arcname باعث می‌شود ساختار پوشه در فایل zip حفظ شود
                    zipf.write(file_path, arcname=file_path)

            print(f"پوشه '{SUBMISSION_DIR}' با موفقیت اضافه شد.")

            # ب) اضافه کردن فایل نوت‌بوک
            if os.path.exists(NOTEBOOK_FILE):
                print(f"در حال اضافه کردن فایل نوت‌بوک '{NOTEBOOK_FILE}'...")
                zipf.write(NOTEBOOK_FILE)
                print(f"فایل '{NOTEBOOK_FILE}' با موفقیت اضافه شد.")
            else:
                print(f"هشدار: فایل نوت‌بوک '{NOTEBOOK_FILE}' یافت نشد و به فایل فشرده اضافه نشد.")

        print("\n---------------------------------------------------------")
        print(f"✅ فایل '{OUTPUT_ZIP_FILE}' با موفقیت ساخته شد و آماده ارسال است.")
        print("---------------------------------------------------------")

    except Exception as e:
        print(f"\nخطایی در حین ساخت فایل فشرده رخ داد: {e}")


پوشه 'submission' با موفقیت یافت شد.
شروع فرآیند فشرده‌سازی...
در حال اضافه کردن پوشه 'submission'...
پوشه 'submission' با موفقیت اضافه شد.
در حال اضافه کردن فایل نوت‌بوک 'notebook.ipynb'...
فایل 'notebook.ipynb' با موفقیت اضافه شد.

---------------------------------------------------------
✅ فایل 'result.zip' با موفقیت ساخته شد و آماده ارسال است.
---------------------------------------------------------
